In [6]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path


In [7]:

REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}


def read_source_links(txt_path: str) -> list:
    """
    Reads metric page URLs from a text file.
    """
    file = Path(txt_path)
    if not file.exists():
        return []

    return [
        line.strip()
        for line in file.read_text(encoding="utf-8").splitlines()
        if line.strip().startswith("http")
    ]


def fetch_country_master() -> pd.DataFrame:
    """
    Extracts country names and global ranks from the main listing page.
    """
    url = "https://www.globalfirepower.com/countries-listing.php"
    html = requests.get(url, headers=REQUEST_HEADERS, timeout=20).text
    soup = BeautifulSoup(html, "lxml")

    records = []

    for container in soup.find_all("div", class_="recordsetContainer"):
        try:
            country = container.select_one("span.textShadow").get_text(strip=True)
            rank = container.select_one("span.textBold").get_text(strip=True)
            records.append({"Country": country, "Global_Rank": rank})
        except AttributeError:
            continue

    return pd.DataFrame(records)


def parse_metric_page(metric_url: str) -> pd.DataFrame:
    """
    Parses a single metric page and returns Country + Metric column.
    """
    response = requests.get(metric_url, headers=REQUEST_HEADERS, timeout=20)
    soup = BeautifulSoup(response.text, "lxml")

    metric_key = (
        metric_url.rsplit("/", 1)[-1]
        .replace(".php", "")
        .replace("-", "_")
    )

    rows = []

    for row in soup.find_all("div", class_="recordsetContainer"):
        spans = row.find_all("span", class_="textLarge")
        if len(spans) < 2:
            continue

        try:
            country = row.select_one("span.textShadow").get_text(strip=True)
            value = spans[-1].get_text(strip=True)
            rows.append({"Country": country, metric_key: value})
        except AttributeError:
            continue

    return pd.DataFrame(rows)


def normalize_numeric_fields(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans numeric columns by removing commas and extracting numbers.
    """
    for col in df.columns:
        if col in ("Country", "Global_Rank"):
            continue

        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.extract(r"(\d+(?:\.\d+)?)")[0]
        )

    return df


def build_global_firepower_dataset(url_file: str) -> pd.DataFrame:
    """
    Master pipeline:
    - Load country base
    - Loop metric URLs
    - Merge all metrics
    """
    master_df = fetch_country_master()
    metric_urls = read_source_links(url_file)

    for metric_url in metric_urls:
        try:
            metric_df = parse_metric_page(metric_url)
            if not metric_df.empty:
                master_df = master_df.merge(metric_df, on="Country", how="left")
        except Exception:
            continue

    return normalize_numeric_fields(master_df)


if __name__ == "__main__":
    dataset = build_global_firepower_dataset("/content/links_for_military_data.txt")
    dataset.to_csv("global_firepower_raw_metrics.csv", index=False)
    print("global_firepower_raw_metrics.csv generated successfully")


global_firepower_raw_metrics.csv generated successfully


In [8]:
import pandas as pd
from pathlib import Path


def summarize_csv_shape(csv_path: str) -> None:
    file = Path("/content/global_firepower_raw_metrics.csv")

    if not file.exists():
        print("CSV file not found.")
        return

    df = pd.read_csv(file)

    row_count, column_count = df.shape

    print(f"File analyzed : {file.name}")
    print(f"Total rows    : {row_count}")
    print(f"Total columns : {column_count}")


if __name__ == "__main__":
    summarize_csv_shape("global_firepower_raw_metrics.csv")


File analyzed : global_firepower_raw_metrics.csv
Total rows    : 145
Total columns : 57
